In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
myworkingdirectory = r"C:\Users\Sophie.Kaye\Desktop\NPS Crash Data"
os.chdir(myworkingdirectory)

In [3]:
cds_crash = pd.read_excel('./CDS/Legacy_CDS_v2.xlsx') # updated with more lat/longs from Bryce
cds_crash.shape

(204690, 57)

In [4]:
cds_crash.columns

Index(['OBJECTID_1', 'OBJECTID', 'INCID_NO', 'CASE_NUM', 'PARK_ALPHA',
       'STATE_CODE', 'CRASH_DATE', 'CRASH_TIME', 'RTE_NO', 'RTE_NAME',
       'NODE_DIST_FT', 'NODE_DIST_MI', 'NODE_DIR', 'NODE_NUM', 'LIGHT',
       'WEATHER', 'CRASH_LOCATION', 'SURF_COND', 'CRASH_CLASS', 'VEH_COLL',
       'OBJ_STRUCK', 'ROAD_CHAR', 'CON_FACT1', 'CON_FACT2', 'CON_FACT3',
       'CON_FACT4', 'CON_FACT5', 'CON_FACT6', 'HIT_RUN', 'CATEGORY', 'FATALS',
       'INJURED', 'PED_FAT', 'PED_INJ', 'BIKE_FAT', 'BIKE_INJ', 'PED',
       'CRASH_YEAR', 'COMMENTS', 'ZIPFILE', 'LOCATION', 'PHOTOS_TAKEN',
       'USPP_NPS_VEH_INV', 'PARK_PTY_DEST', 'LOCKED_UPDATE', 'LOCKED_BY_USER',
       'DATA_SRC', 'LATITUDE', 'LONGITUDE', 'MILEPOST', 'IMPORT_DATE',
       'FILE_NAME', 'SAVE_DATE', 'ROUTE_IDENT', 'RIP_CYCLE', 'MP_NODE',
       'SPTL_LOC'],
      dtype='object')

In [5]:
cds_crash.loc[cds_crash['PARK_ALPHA'].isnull()==True].shape[0]
# 3 crashes without park names identified (to be dropped during data cleaning)

3

In [6]:
pd.options.display.max_rows = 10000000
cds_crash.PARK_ALPHA.value_counts().sort_index()
# note 778 crashes with park name "ZZZZ", which is CDS code for unknown

PARK_ALPHA
ABLI        7
ACAD     1419
AGFO        1
ALPO       20
AMIS       94
ANJO        4
ANTI      122
APCO        1
APIS        1
ARCH      306
ARPO        1
ASIS      155
BADL      213
BAND       70
BAWA    28179
BIBE      914
BICA       30
BICY       50
BISC        4
BISO      270
BITH       55
BLCA       42
BLRI     7853
BLUE        2
BOST       58
BOWA       12
BRCA      577
BUFF      110
CABR       16
CACH        6
CACL        1
CACO      416
CAGR        6
CAHA      370
CALO       52
CANA       44
CANY      130
CARE       34
CARL        7
CASA        1
CATO      120
CAVE      119
CAVO        7
CEBR        6
CHAT      226
CHCH      780
CHCU        4
CHIC      402
CHIR       18
CHIS        2
CHOH      420
CIRO        1
CLBA       12
CODA       41
COLM      395
COLO     1661
CONG       10
CORO       10
CRLA      504
CUGA     1019
CUIS        2
CURE       84
CUVA      247
DENA      225
DEPO        1
DETO       17
DEVA      837
DEWA     5394
DINO       53
EDIS        1
EISE     

In [7]:
# for any of the 778 crashes with unknown park names, can park names be assigned geospatially using lat/long like in IMARS?
cds_unknown_park = cds_crash[cds_crash['PARK_ALPHA']=="ZZZZ"]
cds_unknown_park[cds_unknown_park['LATITUDE'].isnull()==False].shape[0] # nope!

0

In [8]:
cds_vehicle = pd.read_excel('./CDS/ALL_UNIT.xlsx')

In [9]:
cds_vehicle.loc[cds_vehicle['INCID_NO'].isnull()==True].shape[0]
# no vehicle reports with missing record numbers

0

In [10]:
cds_passenger = pd.read_excel('./CDS/ALL_PASSENGER.xlsx')

KeyboardInterrupt: 

In [ ]:
cds_passenger.loc[cds_passenger['INCID_NO'].isnull()==True].shape[0]
# no person reports with missing record numbers

# Data Cleaning

## Remove crashes missing record numbers

In [ ]:
cds_crash = cds_crash.dropna(subset=['INCID_NO']) 
# not needed for passenger or vehicle tables as shown in tests above

## Drop duplicates

In [ ]:
cds_crash = cds_crash.drop_duplicates(subset=['INCID_NO']) 
# Cannot remove duplicates from vehicle and passenger tables because they are unique records for each person/car involved in crash

## Revert flipped lat/long

In [ ]:
need_revert = cds_crash['LATITUDE'].abs() > 70
cds_crash.loc[need_revert, ['LATITUDE', 'LONGITUDE']] = (cds_crash.loc[need_revert, ['LONGITUDE', 'LATITUDE']].values)

## Adjust signs to correct hemisphere

In [ ]:
cds_crash.loc[cds_crash.LATITUDE < 0,"LATITUDE"] = cds_crash['LATITUDE']*(-1)
cds_crash.loc[cds_crash.LONGITUDE > 0,"LONGITUDE"] = cds_crash['LONGITUDE']*(-1)

## Correct order of magnitude

In [ ]:
cds_crash.loc[cds_crash['LONGITUDE'].abs() > 180, "LONGITUDE"] = cds_crash['LONGITUDE']/10
cds_crash.loc[cds_crash['LATITUDE'].abs() > 80, "LATITUDE"] = cds_crash['LATITUDE']/10

In [ ]:
# for any of the 3 crashes with null park names, can park names be assigned geospatially using cleaned lat/long like in IMARS?
cds_null_park = cds_crash[cds_crash['PARK_ALPHA'].isnull()==True]
cds_null_park[cds_null_park['LATITUDE'].isnull()==False].shape[0] # nope!

## Create new region column

In [ ]:
park_info = pd.read_csv("./Reference Data/Park_Info_Table.csv")

In [ ]:
park_info.columns

In [ ]:
park_info = park_info.rename(columns={'UNIT_CODE':'Park','REGION':'RGN'})
cds_crash = cds_crash.rename(columns={'PARK_ALPHA' : 'Park'})

In [ ]:
# add RGN column from lookup table to CDS crash database, joining the two datasets based on park name
cds_crash = pd.merge(cds_crash, park_info[['RGN','Park']], how='left', on='Park')
# resulting dataframe after join should have one additional column and no additional rows
cds_crash = cds_crash.drop_duplicates(subset=['INCID_NO']) 
cds_crash.shape

In [ ]:
cds_crash.loc[cds_crash['RGN'].isnull()==True].shape[0]
# all crashes were assigned a region

In [ ]:
cds_crash.RGN.value_counts().sort_index()

In [ ]:
# drop rows without region
cds_crash = cds_crash[cds_crash["RGN"].str.contains("Unknown")==False]
cds_crash.RGN.value_counts().sort_index()

In [ ]:
cds_crash.to_csv("CDS_CrashTable_RegionAdded_Clean.csv", index=False)

## Create new urban/rural column

In [ ]:
# add Urban column from lookup table to CDS crash database, joining the two datasets based on park name
cds_crash = pd.merge(cds_crash, park_info[['Urban','Park']], how='left', on='Park')
# resulting dataframe after join should have one additional column and no additional rows
cds_crash = cds_crash.drop_duplicates() 
cds_crash.shape

In [ ]:
cds_crash.head()

In [ ]:
# check to see if any crashes were not assigned an urban/rural designation
cds_crash.loc[cds_crash['Urban'].isnull()].shape[0]

In [ ]:
cds_crash['Urban'].value_counts()

In [ ]:
cds_crash.to_csv("CDS_CrashTable_Region_Urban_Added_Clean.csv", index= False)

# Filter for Necessary Fields, Group by INCID_NO

## Crash Table

In [ ]:
cds_crash['CRASH_TIME'].value_counts().sort_index()
# order if statements in for loop from most to least common to increase speed

In [ ]:
cds_crash['CRASH_DATE'].head()

In [ ]:
cds_crash['CRASH_TIME'].head()

In [ ]:
# reformat crash date and time; separate into individual columns
cds_crash = cds_crash.rename(columns={'CRASH_YEAR':'YEAR', 'CRASH_TIME':'TIME'})
cds_crash['CRASH_DATE'] = pd.to_datetime(cds_crash['CRASH_DATE'], format='%Y%m%d %H:%M:%S.%f')
cds_crash['MONTH'] = cds_crash['CRASH_DATE'].dt.strftime('%m')
cds_crash['DOW'] = cds_crash['CRASH_DATE'].dt.strftime('%w')
cds_crash['HOUR'] = 0 # initialize column with dummy data to replace with for loop
length = range(cds_crash.shape[0])
for i in length:
    if cds_crash['TIME'].iloc[i] > 999 and cds_crash['TIME'].iloc[i] < 2400:
        cds_crash['HOUR'].iloc[i] = int(str(cds_crash['TIME'].iloc[i])[:2]) # parse first two digits from afternoon data in 24-hour timestamp
    elif cds_crash['TIME'].iloc[i] > 99 and cds_crash['TIME'].iloc[i] < 1000:
        cds_crash['HOUR'].iloc[i] = int(str(cds_crash['TIME'].iloc[i])[:1]) # parse first digit from morning data in 24-hour timestamp
    else:
        cds_crash['HOUR'].iloc[i] = '0'
cds_crash_slim = cds_crash[['INCID_NO','LATITUDE','LONGITUDE','Park','RGN','YEAR','MONTH','DOW','HOUR','TIME','Urban']]
cds_crash_slim.head() # looks good!

In [ ]:
cds_crash_slim['HOUR'].value_counts() # to be sure that range (0-23) matches that of IMARS

In [ ]:
cds_crash_slim.shape

In [ ]:
cds_crash_slim.to_csv('./CDS_crash_slim.csv',index=False)

## Crash Details Table

In [ ]:
cds_crash_details_slim = cds_crash[['INCID_NO']]

In [ ]:
cds_crash['CRASH_CLASS'].value_counts()

In [ ]:
# print sums of OBJ STRUCK codes only for crashes where CRASH_CLASS==2
fixed_obj = cds_crash.loc[cds_crash['CRASH_CLASS']=='Collision with Fixed Object']
fixed_obj['OBJ_STRUCK'].value_counts()

In [ ]:
# change strings to match IMARS (in order of from most to least common to increase speed)
cds_crash_details_slim['First_Harmful_Event'] = "0" # initialize column with dummy data to replace with for loop
length = range(cds_crash.shape[0])
for i in length:
    if cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Other Motor Vehicle': # 101,521
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '21. Motor vehicle in transport' 
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Fixed Object': # 47,715
        if cds_crash['OBJ_STRUCK'].iloc[i]=='Tree/Shrub': # 15,497
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '52. Tree/shrub'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Other Fixed Object': # 6,738
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '66. Other fixed object'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Guardrail/Barrier': # 5,984
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '47. Guardrail/cable face'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Rock/Stone Wall': # 5,024
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '50. Wall'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Ditch': # 2,538
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '45. Ditch'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Pole': # 2,383
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '53. Utility pole/light support'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Sign': # 2,314
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '54. Traffic sign support'
        elif cds_crash['OBJ_STRUCK'].iloc[i] in (['Embankment','Backslope']): # 1,801 (1787+14)
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '46. Earth embankment/berm' 
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Boulder': # 1,707
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '60. Rock, boulder, rock slide'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Bridge Structure': # 1,188
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '101. Bridge Structure'
        elif cds_crash['OBJ_STRUCK'].iloc[i]=='Barricade': # 1,010
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '51. Other traffic barrier/barricade'
        elif cds_crash['OBJ_STRUCK'].iloc[i] in (['Culvert End Wall','Drainage Structure']): # 845 (461+384)
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '43. End of drainage pipe/structure/culvert'
        elif cds_crash['OBJ_STRUCK'].iloc[i] in (['Unknown', 'Blank On Form', 'Not Applicable']): # 685 (10+8+667)
            cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '99. Unknown'        
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Animal': # 18,927
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '36. Other wild animal (crocodile, birds, coyote, etc.)'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Non-collision': # 14,132
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '16. Other non-collision'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Parked Motor Vehicle': # 8,133
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '22. Parked motor vehicle'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='ColliCsion with Other Object': # 5,383
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '25. Other non-fixed object'
    elif cds_crash['CRASH_CLASS'].iloc[i] in (["Unknown", "Blank On Form"]): # 3,671
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '99. Unknown'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Pedalcycle (Bicycle)': # 1,493
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '18. Bicycle'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Pedestrian': # 1,428
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '17. Pedestrian' 
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Non-Collision(Overturn)': # 34
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '01. Overturn/rollover'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Railway Train': # 32
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '20. Railway vehicle'  
    else:
        cds_crash_details_slim['First_Harmful_Event'].iloc[i] = '99. Unknown'

cds_crash_details_slim['First_Harmful_Event'].value_counts()

In [ ]:
cds_crash.loc[cds_crash['CRASH_CLASS'].isnull()==True].shape[0]

In [ ]:
# "99. Unknown" attribute combines OBJ STRUCK, CRASH CLASS - looks good!
10+8+2849+822+1440+667

In [ ]:
# "0" in First_Harmful_Event sum correspond to sum of OBJ STRUCK codes in fixed_obj that are not mapped to IMARS First_Harmful_Event in schema
47715-15497-6738-5984-5024-2538-2383-2314-1801-1707-1188-1010-845-685

In [ ]:
# change strings to match IMARS (in order of from most to least common to increase speed)
cds_crash_details_slim['First_Harmful_Event_Type'] = "0" # initialize column with dummy data to replace with for loop
length = range(cds_crash.shape[0])
for i in length:
    if cds_crash['CRASH_CLASS'].iloc[i] in (['Collision with Other Motor Vehicle','Collision with Pedestrian','Collision with Pedalcycle (Bicycle)','Collision with Parked Motor Vehicle','Collision with Railway Train']):
        cds_crash_details_slim['First_Harmful_Event_Type'].iloc[i] = 'Collision with person, MV or non-fixed object'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Fixed Object':
        cds_crash_details_slim['First_Harmful_Event_Type'].iloc[i] = 'Collision with fixed object'
    elif cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Animal':
        cds_crash_details_slim['First_Harmful_Event_Type'].iloc[i] = 'Collision with animals'
    elif cds_crash['CRASH_CLASS'].iloc[i] in (['Non-collision','Non-Collision(Overturn)']):
        cds_crash_details_slim['First_Harmful_Event_Type'].iloc[i] = 'Non-collision'   
    else:
        cds_crash_details_slim['First_Harmful_Event_Type'].iloc[i] = 'Unknown'

cds_crash_details_slim['First_Harmful_Event_Type'].value_counts()

In [ ]:
# check First Harmful Event Type sums- looks good!
(101521 + 1428 + 1493 + 8133 + 32), (14132 + 34), (5383 + 2849 + 822 + 1440)

In [ ]:
cds_crash['CON_FACT1'].value_counts()

In [ ]:
# set animal indicator
cds_crash_details_slim['Involving Animal'] = 0 # initialize column with dummy data to replace with for loop
length = range(cds_crash.shape[0])
for i in length:
    if cds_crash['CRASH_CLASS'].iloc[i]=='Collision with Animal':
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1
    elif cds_crash['CON_FACT1'].iloc[i]=='Environment: Animal' :
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1
    elif cds_crash['CON_FACT2'].iloc[i]=='Environment: Animal' :
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1
    elif cds_crash['CON_FACT3'].iloc[i]=='Environment: Animal' :
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1 
    elif cds_crash['CON_FACT4'].iloc[i]=='Environment: Animal' :
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1
    elif cds_crash['CON_FACT5'].iloc[i]=='Environment: Animal' :
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1
    elif cds_crash['CON_FACT6'].iloc[i]=='Environment: Animal' :
        cds_crash_details_slim['Involving Animal'].iloc[i] = 1
    else:
        cds_crash_details_slim['Involving Animal'].iloc[i] = 0
# check logic - looks good!
cds_crash_details_slim['Involving Animal'].sum() # (greater value than either CRASH_CLASS or CON_FACT1, so it must be accounting for both) 

In [ ]:
cds_crash['LIGHT'].value_counts()

In [ ]:
cds_crash.loc[cds_crash['LIGHT'].isnull()].shape[0]

In [ ]:
cds_crash_details_slim['Daylight'] = np.where(cds_crash['LIGHT']=="Daylight", 1,0)
cds_crash_details_slim['Dawn'] = np.where(cds_crash['LIGHT']=="Dawn", 1,0)
cds_crash_details_slim['Dusk'] = np.where(cds_crash['LIGHT']=="Dusk", 1,0)
cds_crash_details_slim['Dark_Lit'] = np.where(cds_crash['LIGHT']=="Dark - Lighted", 1,0)
cds_crash_details_slim['Dark_NotLit'] = np.where(cds_crash['LIGHT']=="Dark - Not Lighted", 1,0)
cds_crash_details_slim['Dark_UnknownLit'] = 0
cds_crash_details_slim['UnknownLit'] = np.where(cds_crash['LIGHT']=="Unknown", 1,0)
cds_crash_details_slim['BlankLit'] = np.where(cds_crash['LIGHT']=="Blank On Form", 1,0)
cds_crash_details_slim['NullLit'] = np.where(pd.isna(cds_crash['LIGHT']),1,0)
cds_crash_details_slim['UnknownBlankNullLit'] = cds_crash_details_slim['UnknownLit']+cds_crash_details_slim['BlankLit']+cds_crash_details_slim['NullLit'] # combined field for which unspecified lighting attribute may be able to be assigned based on sun location with respect to time
# check sum
cds_crash_details_slim['Daylight'].sum(), cds_crash_details_slim['Dawn'].sum(), cds_crash_details_slim['Dusk'].sum(), cds_crash_details_slim['Dark_Lit'].sum(), cds_crash_details_slim['Dark_NotLit'].sum(), cds_crash_details_slim['Dark_UnknownLit'].sum(), cds_crash_details_slim['UnknownLit'].sum(), cds_crash_details_slim['BlankLit'].sum(), cds_crash_details_slim['NullLit'].sum(), cds_crash_details_slim['UnknownBlankNullLit'].sum()

In [ ]:
# replace unspecified Lighting attributes with more accurate data based on general sun position with respect to time
# times from NOAA: https://vividmaps.com/sunrise-time-sunset-time-and-hours-of/ and https://www.google.com/search?sca_esv=572573644&q=sunrise+and+sunset+on+winter+solstice+map&tbm=isch&source=lnms&sa=X&ved=2ahUKEwjgnaaU9O6BAxUlFFkFHeLJDn8Q0pQJegQIDRAB&biw=1912&bih=966&dpr=1

length = range(cds_crash.shape[0]) # search through all rows in dataset
for i in length: # assign lighting condition based on crash hour when unknown
    if cds_crash["RGN"].iloc[i] == "NCR": # NCR has more precise guaranteed lighting times due to concentrated crash locations
        if cds_crash_details_slim['UnknownBlankNullLit'].iloc[i] == 1:
            if cds_crash['TIME'].iloc[i] > 829 and cds_crash['TIME'].iloc[i] < 1531: # guaranteed daylight period w/ hour buffer
                cds_crash_details_slim['Daylight'].iloc[i] = 1
                cds_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
            elif cds_crash['TIME'].iloc[i] > 2159 or cds_crash['TIME'].iloc[i] < 431: # guaranteed dark period w/ hour buffer 
                cds_crash_details_slim['Dark_UnknownLit'].iloc[i] = 1
                cds_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
    else: # all other regions with less precise guaranteed lighting times due to spread crash locations
        if cds_crash_details_slim['UnknownBlankNullLit'].iloc[i] == 1:
            if cds_crash['TIME'].iloc[i] > 959 and cds_crash['TIME'].iloc[i] < 1431:
                cds_crash_details_slim['Daylight'].iloc[i] = 1
                cds_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0
            elif cds_crash['TIME'].iloc[i] > 2259 or cds_crash['TIME'].iloc[i] < 331:
                cds_crash_details_slim['Dark_UnknownLit'].iloc[i] = 1
                cds_crash_details_slim['UnknownBlankNullLit'].iloc[i] = 0

# check sum
cds_crash_details_slim['Daylight'].sum(), cds_crash_details_slim['Dawn'].sum(), cds_crash_details_slim['Dusk'].sum(), cds_crash_details_slim['Dark_Lit'].sum(), cds_crash_details_slim['Dark_NotLit'].sum(), cds_crash_details_slim['Dark_UnknownLit'].sum(), cds_crash_details_slim['UnknownBlankNullLit'].sum()

In [ ]:
# group adverse lighting attributes into bin for poor visibility
PoorLighting = cds_crash_details_slim[['Dawn','Dusk','Dark_NotLit','Dark_Lit','Dark_UnknownLit']]
cds_crash_details_slim['Poor Lighting'] = [1 if x > 0 else 0 for x in np.sum(PoorLighting.values == 1, 1)] # assign value of 1 in new "poor lighting" column if flags are set for at least one darkness indicator 
cds_crash_details_slim['Poor Lighting'].sum(), (14015+6429+2757+1764+12022) # check logic - looks good!

In [ ]:
#cds_crash['WEATHER'].value_counts()

In [ ]:
#cds_crash['SURF_COND'].value_counts()

In [ ]:
## set flags for adverse visibility conditions due to precip
#cds_crash_details_slim['Bad Weather'] = np.where(cds_crash['WEATHER'].isin(['Rain','Snow','Fog, Smog, Smoke','Sleet, Hail, Freezing Rain','Blowing Sand, Soil, Dirt, Etc.'])==True, 1,0)
#cds_crash_details_slim['Bad Road Condition'] = np.where(cds_crash['SURF_COND'].isin(['Wet','Icy or Slushy','Snowy'])==True, 1,0)
#cds_crash_details_slim['Bad Circumstance 1'] = np.where(cds_crash['CON_FACT1'].isin(['Road: Wet','Road: Icy or Slushy','Environment: Smog, Smoke','Environment: Sleet, Hail, Freezing Rain','Environment: Blowing Sand, Soil, Dirt','Environment: Rain, Snow'])==True, 1,0)
#cds_crash_details_slim['Bad Circumstance 2'] = np.where(cds_crash['CON_FACT2'].isin(['Road: Wet','Road: Icy or Slushy','Environment: Smog, Smoke','Environment: Sleet, Hail, Freezing Rain','Environment: Blowing Sand, Soil, Dirt','Environment: Rain, Snow'])==True, 1,0)
#cds_crash_details_slim['Bad Circumstance 3'] = np.where(cds_crash['CON_FACT3'].isin(['Road: Wet','Road: Icy or Slushy','Environment: Smog, Smoke','Environment: Sleet, Hail, Freezing Rain','Environment: Blowing Sand, Soil, Dirt','Environment: Rain, Snow'])==True, 1,0)
#cds_crash_details_slim['Bad Circumstance 4'] = np.where(cds_crash['CON_FACT4'].isin(['Road: Wet','Road: Icy or Slushy','Environment: Smog, Smoke','Environment: Sleet, Hail, Freezing Rain','Environment: Blowing Sand, Soil, Dirt','Environment: Rain, Snow'])==True, 1,0)
#cds_crash_details_slim['Bad Circumstance 5'] = np.where(cds_crash['CON_FACT5'].isin(['Road: Wet','Road: Icy or Slushy','Environment: Smog, Smoke','Environment: Sleet, Hail, Freezing Rain','Environment: Blowing Sand, Soil, Dirt','Environment: Rain, Snow'])==True, 1,0)
#cds_crash_details_slim['Bad Circumstance 6'] = np.where(cds_crash['CON_FACT6'].isin(['Road: Wet','Road: Icy or Slushy','Environment: Smog, Smoke','Environment: Sleet, Hail, Freezing Rain','Environment: Blowing Sand, Soil, Dirt','Environment: Rain, Snow'])==True, 1,0)

In [ ]:
## bad weather sum check - looks good!
#(22786+5753+1897+1421+192), cds_crash_details_slim['Bad Weather'].sum()

In [ ]:
## bad road condition sum check - looks good!
#(33773+8639+5028), cds_crash_details_slim['Bad Road Condition'].sum()

In [ ]:
## bad circumstance sum check - looks good!
#(6237+2201+1804+106+136+803), cds_crash_details_slim['Bad Circumstance 1'].sum()

In [ ]:
## check value to order from greatest to least to speed up following for loop
#cds_crash_details_slim['Bad Circumstance 1'].sum(), cds_crash_details_slim['Bad Circumstance 2'].sum(), cds_crash_details_slim['Bad Circumstance 3'].sum(), cds_crash_details_slim['Bad Circumstance 4'].sum(), cds_crash_details_slim['Bad Circumstance 5'].sum(), cds_crash_details_slim['Bad Circumstance 6'].sum()

In [ ]:
## group adverse weather attributes into bin for poor visibility
#AdverseWeather = cds_crash_details_slim[['Bad Weather','Bad Road Condition','Bad Circumstance 1','Bad Circumstance 2','Bad Circumstance 3','Bad Circumstance 4','Bad Circumstance 5','Bad Circumstance 6']]
#cds_crash_details_slim['Adverse Weather'] = [1 if x > 0 else 0 for x in np.sum(AdverseWeather.values == 1, 1)] # assign value of 1 in new "adverse weather" column if flags are set for at least one bad weather indicator 
#cds_crash_details_slim['Adverse Weather'].sum() # (greater value than either Bad Road Condition, Bad Weather, or Bad Circumstance 1, so it must be accounting for all) 

In [ ]:
cds_crash_details_slim.columns

In [ ]:
cds_crash_details_slim = cds_crash_details_slim.drop(columns={'UnknownLit','BlankLit','NullLit'})#'Bad Weather','Bad Road Condition','Bad Circumstance 1','Bad Circumstance 2', 'Bad Circumstance 3','Bad Circumstance 4','Bad Circumstance 5','Bad Circumstance 6'])
cds_crash_details_slim.columns

In [ ]:
cds_crash_details_slim.shape

In [ ]:
cds_crash_details_slim.to_csv('./CDS_crash_details_slim.csv',index=False)

## Passenger Table

In [ ]:
# determine number of driver injuries for each crash
cds_vehicle_inj = cds_vehicle[['INCID_NO','PED_TYPE']] # subset needed columns (PED_TYPE for Num_Ped_Deaths later calc)
cds_vehicle_inj['No Injury'] = np.where(cds_vehicle['DRIVER_INJ'] == 0.0, 1, 0)
cds_vehicle_inj['Possible Injury'] = np.where(cds_vehicle['DRIVER_INJ'] == 1.0, 1, 0)
cds_vehicle_inj['Non-incapacitating Injury'] = np.where(cds_vehicle['DRIVER_INJ'] == 2.0, 1, 0)
cds_vehicle_inj['Incapacitating Injury'] = np.where(cds_vehicle['DRIVER_INJ'] == 3.0, 1, 0)
cds_vehicle_inj['Fatality'] = np.where(cds_vehicle['DRIVER_INJ'] == 4.0, 1, 0)
# collapse multiple rows for each person involved in the crash into a single row for each crash
#injuries/fatalities and number of people involved should be summed as total numbers per crash ("sum" function)
cds_vehicle_inj_agg = cds_vehicle_inj.groupby(by=['INCID_NO']).sum().reset_index()
cds_vehicle_inj_agg.shape

In [ ]:
# determine number of passenger injuries for each crash
cds_passenger_inj = cds_passenger[['INCID_NO']] # subset needed columns
cds_passenger_inj['No Injury Mod'] = np.where(cds_passenger['PASS_INJ'] == 0.0, 1, 0)
cds_passenger_inj['Possible Injury Mod'] = np.where(cds_passenger['PASS_INJ'] == 1.0, 1, 0)
cds_passenger_inj['Non-incapacitating Injury Mod'] = np.where(cds_passenger['PASS_INJ'] == 2.0, 1, 0)
cds_passenger_inj['Incapacitating Injury Mod'] = np.where(cds_passenger['PASS_INJ'] == 3.0, 1, 0)
cds_passenger_inj['Fatality Mod'] = np.where(cds_passenger['PASS_INJ'] == 4.0, 1, 0)
# collapse multiple rows for each person involved in the crash into a single row for each crash
#injuries/fatalities and number of people involved should be summed as total numbers per crash ("sum" function)
cds_passenger_inj_agg = cds_passenger_inj.groupby(by=['INCID_NO']).sum().reset_index()
cds_passenger_inj_agg.shape

In [ ]:
# join driver and passenger injuries for each crash
merged_injury_agg = cds_vehicle_inj_agg.merge(cds_passenger_inj_agg, how='outer', on='INCID_NO') # left join to retain records in larger vehicle_inj_agg table
merged_injury_agg.shape

In [ ]:
merged_injury_agg.head()

In [ ]:
merged_injury_agg = merged_injury_agg.fillna(0) # required to make subsequent column addition step work (cannot handle NaN)
merged_injury_agg.head()

In [ ]:
# combine driver and passenger injuries to single value for each crash
cds_combined_injury_agg = merged_injury_agg[['INCID_NO']] # subset needed columns
cds_combined_injury_agg['No Injury'] = merged_injury_agg['No Injury'] + merged_injury_agg['No Injury Mod']
cds_combined_injury_agg['Possible Injury'] = merged_injury_agg['Possible Injury'] + merged_injury_agg['Possible Injury Mod']
cds_combined_injury_agg['Non-incapacitating Injury'] = merged_injury_agg['Non-incapacitating Injury'] + merged_injury_agg['Non-incapacitating Injury Mod']
cds_combined_injury_agg['Incapacitating Injury'] = merged_injury_agg['Incapacitating Injury'] + merged_injury_agg['Incapacitating Injury Mod']
cds_combined_injury_agg['Fatality'] = merged_injury_agg['Fatality'] + merged_injury_agg['Fatality Mod']
# sum of driver and passenger total injuries looks good!!
cds_combined_injury_agg['Fatality'].sum(), cds_vehicle_inj_agg['Fatality'].sum()+cds_passenger_inj_agg['Fatality Mod'].sum()

In [ ]:
cds_combined_injury_agg['Crash_Severity'] = 0 # initialize column with dummy data to replace with for loop
length = range(cds_combined_injury_agg.shape[0]) # search through all rows in aggregated dataset (to examine all injuries from each crash as a whole)
for i in length: # determine most severe injury outcome
    if cds_combined_injury_agg['Fatality'].iloc[i] > 0:
        cds_combined_injury_agg['Crash_Severity'].iloc[i] = 'Fatal'
    elif cds_combined_injury_agg['Incapacitating Injury'].iloc[i] > 0:
        cds_combined_injury_agg['Crash_Severity'].iloc[i] = 'Incap'
    elif cds_combined_injury_agg['Non-incapacitating Injury'].iloc[i] > 0:
        cds_combined_injury_agg['Crash_Severity'].iloc[i] = 'Non-Incap'
    elif cds_combined_injury_agg['Possible Injury'].iloc[i] > 0:
        cds_combined_injury_agg['Crash_Severity'].iloc[i] = 'Possible'
    else:
        cds_combined_injury_agg['Crash_Severity'].iloc[i] = 'No Inj'

#set flags for each Crash Severity attribute
cds_combined_injury_agg['CrashSeverity_Fatal'] = np.where(cds_combined_injury_agg['Crash_Severity']=='Fatal', 1,0)
cds_combined_injury_agg['CrashSeverity_Incap'] = np.where(cds_combined_injury_agg['Crash_Severity']=='Incap', 1,0)
cds_combined_injury_agg['CrashSeverity_NonIncap'] = np.where(cds_combined_injury_agg['Crash_Severity']=='Non-Incap', 1,0)
cds_combined_injury_agg['CrashSeverity_Possible'] = np.where(cds_combined_injury_agg['Crash_Severity']=='Possible', 1,0)
cds_combined_injury_agg['CrashSeverity_NoInj'] = np.where(cds_combined_injury_agg['Crash_Severity']=='No Inj', 1,0)

In [ ]:
# test crash severity logic - looks good!
test=cds_combined_injury_agg.loc[cds_combined_injury_agg['Fatality']==0] # remove fatal crashes
cds_combined_injury_agg['CrashSeverity_Incap'].sum(), test.loc[test['Incapacitating Injury']>0].shape[0] # number of crashes with incapacitating severity = number of crashes resulting in incapacitating injuries when fatal crashes are removed from dataset

In [ ]:
cds_vehicle['PED_TYPE'].value_counts().sort_index()

In [ ]:
cds_vehicle.columns

In [ ]:
# determine number of people involved
cds_passenger_slim = cds_vehicle[['INCID_NO']] # subset needed columns
cds_passenger_slim['Num_Motorist'] = np.where(cds_vehicle['PED_TYPE'].isin([1.0, 2.0, 3.0, 4.0])==False, cds_vehicle['NUM_OCC'], 0)
cds_passenger_slim['Num_Non_Motorist'] = np.where(cds_vehicle['PED_TYPE'].isin([1.0, 2.0, 3.0, 4.0])==True, 1, 0)
cds_passenger_slim['Num_Pedestrian'] = np.where(cds_vehicle['PED_TYPE']==1.0, 1,0)
# check new non-motorist logic, looks good!
cds_passenger_slim['Num_Non_Motorist'].sum(), (1045+1039+6+6) # combined number of Pedestrians, Pedalcyclists, Riders of Animals, and Riders of Animal-drawn Vehicle

In [ ]:
# check new motorist logic - looks good!
test = cds_vehicle.loc[cds_vehicle['PED_TYPE'].isin([1.0, 2.0, 3.0, 4.0])==True]
cds_passenger_slim['Num_Motorist'].sum(), cds_vehicle['NUM_OCC'].sum()-test['NUM_OCC'].sum() # number of motorists = all people - number of non-motorists 

In [ ]:
cds_passenger_slim['Num_Pedestrian'].sum() # exactly as expected from 'PED_TYPE' value counts

In [ ]:
cds_passenger_slim['Num_Ped_Deaths'] = np.where(cds_vehicle_inj['PED_TYPE']==1.0, cds_vehicle_inj['Fatality'], 0)
# test logic, looks good!
test = cds_vehicle_inj.loc[cds_vehicle_inj['PED_TYPE']==1.0]
cds_passenger_slim['Num_Ped_Deaths'].sum(), test['Fatality'].sum() # number of pedestrian deaths = "driver" fatalities in subset of vehicle reports for pedestrians

In [ ]:
# collapse multiple rows for each person involved in the crash into a single row for each crash
# number of people involved should be summed as total numbers per crash ("sum" function)
cds_passenger_slim_agg = cds_passenger_slim.groupby(by=['INCID_NO']).sum().reset_index()
cds_passenger_slim_agg.shape

In [ ]:
cds_passenger_slim_agg['Num_Pedestrian'].value_counts()

In [ ]:
# set binary indicators for VRU and Pedestrian
cds_passenger_slim_agg['Pedestrian'] = np.where(cds_passenger_slim_agg['Num_Pedestrian'] > 0, 1, 0)
cds_passenger_slim_agg['VRU'] = np.where(cds_passenger_slim_agg['Num_Non_Motorist'] > 0, 1, 0)
# test logic - looks good!
cds_passenger_slim_agg['Pedestrian'].sum(), (937+45+3+1+1) # replacing number of pedestrians involved in each crash with 1 for all crashes involving pedestrians

In [ ]:
cds_passenger_slim_agg.shape, cds_combined_injury_agg.shape

In [ ]:
cds_passenger_slim_agg.columns

In [ ]:
cds_combined_injury_agg.columns

In [ ]:
# combine injuries with number of people involved
cds_passenger_slim_agg = cds_passenger_slim_agg.merge(cds_combined_injury_agg, how='left', on='INCID_NO')
cds_passenger_slim_agg.shape

In [ ]:
cds_passenger_slim_agg.columns

In [ ]:
# reorder columns to match IMARS
cds_passenger_slim_agg = cds_passenger_slim_agg[['INCID_NO','No Injury','Possible Injury','Non-incapacitating Injury','Incapacitating Injury','Fatality','Num_Motorist','Num_Non_Motorist','Num_Pedestrian','Num_Ped_Deaths','Pedestrian','VRU','Crash_Severity','CrashSeverity_Fatal','CrashSeverity_Incap','CrashSeverity_NonIncap','CrashSeverity_Possible','CrashSeverity_NoInj']]

In [ ]:
cds_passenger_slim_agg.to_csv("./CDS_passenger_slim_agg.csv",index=False)

## Vehicle Table

In [ ]:
cds_vehicle['SPEED_LIMIT'].value_counts()

In [ ]:
cds_vehicle.PED_LOC.value_counts().sort_index()

In [ ]:
# number of vehicle entries in which a crash occurred at a known site other than a crosswalk 
# (i.e., "in roadway (not in crosswalk), "on trail/bikeway (off roadway)", "other off roadway", "other"):
1195+79+199+1170

In [ ]:
cds_vehicle_slim = cds_vehicle[['INCID_NO']] # subset for needed columns
cds_vehicle_slim['5_mph']= np.where(cds_vehicle['SPEED_LIMIT']==5.0, 1,0)
cds_vehicle_slim['10_mph']= np.where(cds_vehicle['SPEED_LIMIT']==10.0, 1,0)
cds_vehicle_slim['15_mph']= np.where(cds_vehicle['SPEED_LIMIT']==15.0, 1,0)
cds_vehicle_slim['20_mph']= np.where(cds_vehicle['SPEED_LIMIT']==20.0, 1,0)
cds_vehicle_slim['25_mph']= np.where(cds_vehicle['SPEED_LIMIT']==25.0, 1,0)
cds_vehicle_slim['30_mph']= np.where(cds_vehicle['SPEED_LIMIT']==30.0, 1,0)
cds_vehicle_slim['35_mph']= np.where(cds_vehicle['SPEED_LIMIT']==35.0, 1,0)
cds_vehicle_slim['40_mph']= np.where(cds_vehicle['SPEED_LIMIT']==40.0, 1,0)
cds_vehicle_slim['45_mph']= np.where(cds_vehicle['SPEED_LIMIT']==45.0, 1,0)
cds_vehicle_slim['50_mph']= np.where(cds_vehicle['SPEED_LIMIT']==50.0, 1,0)
cds_vehicle_slim['55_mph']= np.where(cds_vehicle['SPEED_LIMIT']==55.0, 1,0)
cds_vehicle_slim['60_mph']= np.where(cds_vehicle['SPEED_LIMIT']==60.0, 1,0)
cds_vehicle_slim['65_mph']= np.where(cds_vehicle['SPEED_LIMIT']==65.0, 1,0)
cds_vehicle_slim['70_mph']= np.where(cds_vehicle['SPEED_LIMIT']==70.0, 1,0)
cds_vehicle_slim['75_mph']= np.where(cds_vehicle['SPEED_LIMIT']==75.0, 1,0)
cds_vehicle_slim['no_posted_speed']= 0
cds_vehicle_slim['Crosswalk']= np.where(cds_vehicle['PED_LOC']==1.0, 1,0)
cds_vehicle_slim['Outside a Crosswalk']= np.where(cds_vehicle['PED_LOC'].isin([2.0,3.0,4.0,88.0]), 1,0)
cds_vehicle_slim['Outside a Crosswalk'].sum()

In [92]:
# collapse multiple rows for each vehicle involved in the crash into a single row for each crash
# only one speed limit should exist per crash to not double count data attribute ("first" function)
cds_vehicle_slim_agg = cds_vehicle_slim.groupby(by=['INCID_NO']).first().reset_index()
cds_vehicle_slim_agg.shape

(194715, 17)

In [93]:
cds_vehicletypes = cds_vehicle[['INCID_NO']] # subset needed columns

In [94]:
cds_vehicletypes['NUM_VEH'] = np.where(cds_vehicle['PED_TYPE'].isin([1.0, 2.0, 3.0, 4.0])==False, 1, 0) # number of entries in vehicle table, excluding VRUs

<ipython-input-94-5d4e988f18a1>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cds_vehicletypes['NUM_VEH'] = np.where(cds_vehicle['PED_TYPE'].isin([1.0, 2.0, 3.0, 4.0])==False, 1, 0) # number of entries in vehicle table, excluding VRUs


In [95]:
cds_vehicle['BODY_TYPE'].value_counts()

1.0     199904
6.0      31569
3.0      19692
7.0       9761
0.0       7729
4.0       7525
12.0      6409
88.0      5551
15.0      3482
10.0      2732
8.0       2181
99.0      1911
2.0       1721
9.0       1309
11.0       973
13.0       943
16.0       872
5.0        339
17.0       215
14.0       199
30.0        69
18.0        50
50.0        36
98.0        19
Name: BODY_TYPE, dtype: int64

In [96]:
# set flags for each vehicle type
cds_vehicletypes['Num_Motorcycle'] = np.where(cds_vehicle['BODY_TYPE']==4.0, 1,0)
cds_vehicletypes['Pax'] = np.where(cds_vehicle['BODY_TYPE'].isin([1.0,2.0,16.0])==True, 1,0)
cds_vehicletypes['SUV'] = np.where(cds_vehicle['BODY_TYPE']==15.0, 1,0)
cds_vehicletypes['Van'] = np.where(cds_vehicle['BODY_TYPE'].isin([3.0,6.0,17.0])==True, 1,0)
cds_vehicletypes['Bus'] = np.where(cds_vehicle['BODY_TYPE'].isin([12.0,11.0,10.0])==True, 1,0)
cds_vehicletypes['Medium Truck'] = np.where(cds_vehicle['BODY_TYPE']==7.0, 1,0)
cds_vehicletypes['Heavy Truck'] = np.where(cds_vehicle['BODY_TYPE'].isin([8.0,9.0])==True, 1,0)
cds_vehicletypes['Equipment'] = np.where(cds_vehicle['BODY_TYPE'].isin([18.0,30.0,50.0])==True, 1,0)
cds_vehicletypes['Nonroad HighSpeed'] = np.where(cds_vehicle['BODY_TYPE'].isin([14.0,13.0])==True, 1,0)
cds_vehicletypes['Nonroad LowSpeed'] = np.where(cds_vehicle['BODY_TYPE']==5.0, 1,0)
cds_vehicletypes['Unknown99'] = np.where(cds_vehicle['BODY_TYPE'].isin([98.0,99.0])==True, 1,0)
cds_vehicletypes['Empty'] = np.where(pd.isna(cds_vehicle['BODY_TYPE']),1,0)
cds_vehicletypes['Unknown'] = cds_vehicletypes['Unknown99']+cds_vehicletypes['Empty']

<ipython-input-96-e18a3265d130>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cds_vehicletypes['Num_Motorcycle'] = np.where(cds_vehicle['BODY_TYPE']==4.0, 1,0)
<ipython-input-96-e18a3265d130>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cds_vehicletypes['Pax'] = np.where(cds_vehicle['BODY_TYPE'].isin([1.0,2.0,16.0])==True, 1,0)
<ipython-input-96-e18a3265d130>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

In [97]:
cds_vehicletypes = cds_vehicletypes.drop(columns={'Unknown99', 'Empty'}) # drop columns used for calculated fields
# collapse multiple rows for each vehicle involved in the crash into a single row for each crash
# vehicle types and number of people involved should be summed as total numbers per crash ("sum" function)
cds_vehicletypes_agg = cds_vehicletypes.groupby(by=['INCID_NO']).sum().reset_index()
cds_vehicletypes_agg.shape

(194715, 13)

In [98]:
cds_vehicletypes_agg['Num_Motorcycle'].value_counts()

0    187574
1      6784
2       334
3        19
4         4
Name: Num_Motorcycle, dtype: int64

In [99]:
# set indicator for crashes involving motorcycles
cds_vehicletypes_agg['Motorcycle_Ind'] = np.where(cds_vehicletypes_agg['Num_Motorcycle'] > 0, 1,0)
cds_vehicletypes_agg['Motorcycle_Ind'].sum(), (6784+334+19+4) # looks good (replacing number of motorcyclists involved in each crash with 1 for all crashes involving motorcyclists)

(7141, 7141)

In [100]:
cds_vehicletypes_agg.columns

Index(['INCID_NO', 'NUM_VEH', 'Num_Motorcycle', 'Pax', 'SUV', 'Van', 'Bus',
       'Medium Truck', 'Heavy Truck', 'Equipment', 'Nonroad HighSpeed',
       'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind'],
      dtype='object')

In [101]:
# join aggregated ('sum' and 'first') vehicle data into single table
cds_slim_vehicle_agg = cds_vehicle_slim_agg.merge(cds_vehicletypes_agg, how='right', on='INCID_NO')
cds_slim_vehicle_agg = cds_slim_vehicle_agg.drop_duplicates()
cds_slim_vehicle_agg.shape

(194715, 30)

In [102]:
cds_slim_vehicle_agg.columns

Index(['INCID_NO', '5_mph', '10_mph', '15_mph', '20_mph', '25_mph', '30_mph',
       '35_mph', '40_mph', '45_mph', '50_mph', '55_mph', '60_mph', '65_mph',
       '70_mph', '75_mph', 'no_posted_speed', 'NUM_VEH', 'Num_Motorcycle',
       'Pax', 'SUV', 'Van', 'Bus', 'Medium Truck', 'Heavy Truck', 'Equipment',
       'Nonroad HighSpeed', 'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind'],
      dtype='object')

In [103]:
cds_slim_vehicle_agg.to_csv("./CDS_slim_vehicle_agg.csv",index=False)

# Merge Slim Tables

In [104]:
cds_crash_slim.shape, cds_crash_details_slim.shape, cds_passenger_slim_agg.shape, cds_slim_vehicle_agg.shape

((203909, 11), (203909, 12), (194715, 18), (194715, 30))

In [105]:
cds_crash_details_slim_merged = cds_crash_details_slim.merge(cds_crash_slim, how='right', on='INCID_NO')
cds_crash_details_slim_merged.shape

(203909, 22)

In [106]:
cds_crash_details_slim_merged.columns

Index(['INCID_NO', 'First_Harmful_Event', 'First_Harmful_Event_Type',
       'Involving Animal', 'Daylight', 'Dawn', 'Dusk', 'Dark_Lit',
       'Dark_NotLit', 'Dark_UnknownLit', 'UnknownBlankNullLit',
       'Poor Lighting', 'LATITUDE', 'LONGITUDE', 'Park', 'RGN', 'YEAR',
       'MONTH', 'DOW', 'HOUR', 'TIME', 'Urban'],
      dtype='object')

In [125]:
# changed to inner join because crash table and vehicle table have differing lists of INCID_NO such that all other joing produced null columns for certain rows depending on which table the INCID_NO came from
cds_crash_details_and_passenger_slim = cds_passenger_slim_agg.merge(cds_crash_details_slim_merged, how='inner', on='INCID_NO')
cds_crash_details_and_passenger_slim.shape

(194307, 39)

In [126]:
cds_crash_details_and_passenger_slim.columns

Index(['INCID_NO', 'No Injury', 'Possible Injury', 'Non-incapacitating Injury',
       'Incapacitating Injury', 'Fatality', 'Num_Motorist', 'Num_Non_Motorist',
       'Num_Pedestrian', 'Num_Ped_Deaths', 'Pedestrian', 'VRU',
       'Crash_Severity', 'CrashSeverity_Fatal', 'CrashSeverity_Incap',
       'CrashSeverity_NonIncap', 'CrashSeverity_Possible',
       'CrashSeverity_NoInj', 'First_Harmful_Event',
       'First_Harmful_Event_Type', 'Involving Animal', 'Daylight', 'Dawn',
       'Dusk', 'Dark_Lit', 'Dark_NotLit', 'Dark_UnknownLit',
       'UnknownBlankNullLit', 'Poor Lighting', 'LATITUDE', 'LONGITUDE', 'Park',
       'RGN', 'YEAR', 'MONTH', 'DOW', 'HOUR', 'TIME', 'Urban'],
      dtype='object')

In [127]:
#cds_crash_details_and_passenger_slim.to_csv('test.csv', index=False)

In [128]:
cds_slim_all = cds_slim_vehicle_agg.merge(cds_crash_details_and_passenger_slim, how='right', on='INCID_NO')
cds_slim_all['database'] = 'CDS' # add column with database name
cds_slim_all.shape

(194307, 69)

In [129]:
cds_slim_all.columns

Index(['INCID_NO', '5_mph', '10_mph', '15_mph', '20_mph', '25_mph', '30_mph',
       '35_mph', '40_mph', '45_mph', '50_mph', '55_mph', '60_mph', '65_mph',
       '70_mph', '75_mph', 'no_posted_speed', 'NUM_VEH', 'Num_Motorcycle',
       'Pax', 'SUV', 'Van', 'Bus', 'Medium Truck', 'Heavy Truck', 'Equipment',
       'Nonroad HighSpeed', 'Nonroad LowSpeed', 'Unknown', 'Motorcycle_Ind',
       'No Injury', 'Possible Injury', 'Non-incapacitating Injury',
       'Incapacitating Injury', 'Fatality', 'Num_Motorist', 'Num_Non_Motorist',
       'Num_Pedestrian', 'Num_Ped_Deaths', 'Pedestrian', 'VRU',
       'Crash_Severity', 'CrashSeverity_Fatal', 'CrashSeverity_Incap',
       'CrashSeverity_NonIncap', 'CrashSeverity_Possible',
       'CrashSeverity_NoInj', 'First_Harmful_Event',
       'First_Harmful_Event_Type', 'Involving Animal', 'Daylight', 'Dawn',
       'Dusk', 'Dark_Lit', 'Dark_NotLit', 'Dark_UnknownLit',
       'UnknownBlankNullLit', 'Poor Lighting', 'LATITUDE', 'LONGITUDE', 'Park',
   

In [130]:
cds_slim_all.to_csv("./CDS_slim_all_clean.csv",index=False)